# Agentic Default-Payments Pipeline — Demo

End-to-end walkthrough of the CrewAI pipeline:

1. **DataAgent** loads the Taiwan default-of-credit-card-clients CSV and emits a JSON summary.
2. **TrainerAgent** trains Random Forest, XGBoost, and an MLP and emits a JSON metric report.
3. **ExplainerAgent** turns that report into a plain-English interpretability brief.

Set `GEMINI_API_KEY` in your environment (or in `../../../.env`) before running the agent cells.

In [1]:
import os, sys, pathlib, json

PROJECT_ROOT = pathlib.Path().resolve().parents[0]
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT.parents[1] / '.env')
except Exception:
    pass

print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))

GEMINI_API_KEY set: True


In [2]:
PROJECT_ROOT

PosixPath('/home/coder/interpretability-llms-agents/ExploreData/agentic_default_pipeline')

In [3]:
SRC

PosixPath('/home/coder/interpretability-llms-agents/ExploreData/agentic_default_pipeline/src')

## 1. Sanity-check the ML layer (no LLM calls)

Confirm the data loader and trainers work before involving the agents.

In [4]:
from agentic_default.data_loader import load_dataset, csv_to_json_records
from agentic_default.ml_trainer import train_and_evaluate

ds = load_dataset()
print('rows:', ds.metadata.n_rows, '|', 'features:', ds.metadata.n_features)
print('class balance:', ds.metadata.class_balance)
print('feature names:', ds.feature_names[:6], '...')

preview = csv_to_json_records(sample=2)
print('first JSON record:', json.dumps(preview[0], indent=2))

rows: 30000 | features: 23
class balance: {'0': 23364, '1': 6636}
feature names: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0'] ...
first JSON record: {
  "ID": 1,
  "LIMIT_BAL": 20000,
  "SEX": 2,
  "EDUCATION": 2,
  "MARRIAGE": 1,
  "AGE": 24,
  "PAY_0": 2,
  "PAY_2": 2,
  "PAY_3": -1,
  "PAY_4": -1,
  "PAY_5": -2,
  "PAY_6": -2,
  "BILL_AMT1": 3913,
  "BILL_AMT2": 3102,
  "BILL_AMT3": 689,
  "BILL_AMT4": 0,
  "BILL_AMT5": 0,
  "BILL_AMT6": 0,
  "PAY_AMT1": 0,
  "PAY_AMT2": 689,
  "PAY_AMT3": 0,
  "PAY_AMT4": 0,
  "PAY_AMT5": 0,
  "PAY_AMT6": 0,
  "default payment next month": 1
}


In [5]:
report = train_and_evaluate(
    ds.x_train, ds.y_train, ds.x_test, ds.y_test,
    feature_names=ds.feature_names,
    models=['random_forest', 'xgboost', 'neural_network'],
)
for row in report['leaderboard']:
    print(row)
print('best model:', report['best_model'])

{'model_name': 'xgboost', 'roc_auc': 0.7804886446228402, 'f1': 0.4694973157637872, 'precision': 0.6662049861495845, 'recall': 0.3624717407686511, 'accuracy': 0.8188333333333333}
{'model_name': 'neural_network', 'roc_auc': 0.7663087553746765, 'f1': 0.46647230320699706, 'precision': 0.6566347469220246, 'recall': 0.3617181612660136, 'accuracy': 0.817}
{'model_name': 'random_forest', 'roc_auc': 0.7658704439926587, 'f1': 0.4951627088830255, 'precision': 0.5945089757127772, 'recall': 0.4242652599849284, 'accuracy': 0.8086666666666666}
best model: xgboost


## 2. Run the full agentic pipeline

Requires `GEMINI_API_KEY`.

In [6]:
from agentic_default.pipeline import run_pipeline

run = run_pipeline(
    models=['random_forest', 'xgboost', 'neural_network'],
    output_dir=str(PROJECT_ROOT / 'outputs' / 'demo_run'),
)
print('--- Dataset summary ---')
print(json.dumps(run.dataset_summary, indent=2)[:600])
print('\n--- Leaderboard ---')
print(json.dumps(run.metrics_report.get('leaderboard', []), indent=2))

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

: 

In [ ]:
from IPython.display import Markdown
Markdown(run.explanation_markdown)

## 3. Inspect the persisted artifacts

Everything the agents produced is also written to disk for reproducibility.

In [ ]:
out_dir = PROJECT_ROOT / 'outputs' / 'demo_run'
for f in sorted(out_dir.iterdir()):
    print(f.name, '-', f.stat().st_size, 'bytes')